# FLEX — every California NPS unit, all temperature data, all scenarios

Small flex of the library + Coiled stack: grab monthly `T_Max` and `T_Min` for **every NPS unit geographically in California**, across Historical + all three SSPs, 1950–2100, all 15 LOCA2 models. Compute `T_Avg = (T_Max + T_Min) / 2` at the end. Parallel across parks via a 45-worker Coiled cluster.

## A note on map-reduce, Rust, gRPC

Short answer: this is already map-reduce, just wearing Python clothes.

- `build_coiled_task` (in the library) creates a `dask.delayed` task per (park × variable × scenario). That's the **map** step.
- `dask.compute(*tasks)` sends the whole task graph to the scheduler. Dask's scheduler is written in C-extensions + careful Python, and talks to workers over a tuned TCP protocol. Results stream back as they finish — that's the **reduce** step.
- Each worker runs arbitrary Python (xarray, numpy, pandas) with direct S3 access in the same region as the data. The bottleneck is S3 read throughput, not orchestration.

Dropping in a Rust process or a gRPC layer wouldn't help here — the scheduler/worker protocol is already the fastest part, and the per-task work is pandas/xarray ops you can't easily replace with Rust without rewriting the whole pipeline. Where Rust *would* help is per-cell numeric hot loops (e.g., a custom unit-conversion kernel), but we don't have one — Cal-Adapt's Zarr + xarray arithmetic is already vectorized C.

In [ ]:
import os, sys, time
from time import perf_counter
import coiled
import pandas as pd
import matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor, as_completed
from shapely.geometry import Polygon

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, os.path.join(PROJECT_ROOT, "lib"))

from andrewAdaptLibrary import ParkCatalog, CatalogExplorer, get_climate_data


# ---- timing helper ----
class Timer:
    """Context manager: prints start/end and stashes .elapsed for later use."""
    def __init__(self, label): self.label = label
    def __enter__(self):
        self.start = perf_counter()
        print(f"▶ {self.label}…")
        return self
    def __exit__(self, *args):
        self.elapsed = perf_counter() - self.start
        print(f"  ✓ {self.label}: {self.elapsed:.1f}s ({self.elapsed/60:.2f} min)")


T_START = perf_counter()   # notebook-wide wall-clock anchor

NPS_SHP = os.path.join(
    PROJECT_ROOT,
    "USA_National_Park_Service_Lands_20170930_4993375350946852027",
    "USA_Federal_Lands.shp",
)
park_catalog = ParkCatalog(NPS_SHP)
cat = CatalogExplorer(timescale="monthly")       # one shared catalog for all fetches
print(park_catalog, "|", cat)

## 1. Find every CA NPS unit — polygon-contains on the mega layer

Previous version used a loose bounding box and pulled in three Nevada units (Great Basin NP, Lake Mead NRA, Tule Springs). Switching to a hand-crafted California polygon and checking whether each unit's **centroid** is inside it — gives us just California.

In [ ]:
# Rough but decent approximation of the California state outline.
# Vertices are (lon, lat) running clockwise from the NW coast corner.
CA = Polygon([
    (-124.41, 42.00),   # NW: CA-OR corner at Pacific
    (-120.00, 42.00),   # NE top: CA-OR-NV tri-point
    (-120.00, 38.99),   # Lake Tahoe area, CA-NV border kink
    (-114.63, 35.00),   # Colorado River NE: CA-NV-AZ corner
    (-114.63, 32.72),   # CA-AZ border south end
    (-117.12, 32.53),   # CA-MX SW corner at Pacific
    (-117.30, 33.00),   # San Diego coast
    (-118.48, 33.73),   # LA coast
    (-119.88, 34.42),   # Santa Barbara
    (-120.90, 35.27),   # Morro Bay
    (-121.89, 36.60),   # Monterey Bay
    (-122.51, 37.80),   # San Francisco
    (-123.02, 38.30),   # Bodega Bay
    (-123.72, 39.35),   # Mendocino
    (-124.15, 40.44),   # Eureka
    (-124.41, 42.00),   # close loop
])

gdf = park_catalog._gdf                             # internal GDF; TODO lib helper
in_ca_mask = gdf.geometry.centroid.apply(CA.contains)
ca_names = sorted(gdf.loc[in_ca_mask, "unit_name"].unique())
print(f"{len(ca_names)} NPS units with centroid inside California")
for n in ca_names: print(f"  {n}")

## 2. Safety filters — LOCA2 coverage + bbox size

Two checks before dispatching to the cluster:

1. **`_inside_loca2`** — fast geometric check that the park sits inside the LOCA2 3km grid. (All CA units pass this, but keeping the check for any future non-CA variation.)
2. **Bbox ≥ 0.05° in each dim (~5 km)** — LOCA2 grid spacing is ~0.03° (3 km). A park with a bbox smaller than a single cell crashes `rio.clip` with `NoDataInBounds`. This drops tiny urban/historic memorials (Fort Point NHS, Eugene O'Neill NHS, SF Maritime NHP, etc.) — none of which have meaningful climate-scale land coverage anyway.

In [ ]:
def bbox_big_enough(boundary, min_deg=0.05):
    """Park bbox must span at least min_deg in each dim (safe vs 3km LOCA2 grid)."""
    b = boundary.total_bounds
    return (b[2] - b[0]) >= min_deg and (b[3] - b[1]) >= min_deg


boundaries = {n: park_catalog.get_boundary(n) for n in ca_names}

ready   = [n for n, b in boundaries.items()
           if park_catalog._inside_loca2(b) and bbox_big_enough(b)]
skipped = [n for n in ca_names if n not in ready]

print(f"ready: {len(ready)}   skipped: {len(skipped)}")
if skipped:
    print("\nSkipped (outside LOCA2 or bbox too small for 3km grid):")
    for n in skipped:
        b = boundaries[n].total_bounds
        w, h = b[2]-b[0], b[3]-b[1]
        print(f"  ✗ {n:55s}  bbox {w:.3f}° × {h:.3f}°")

## 3. Start the cluster — 45 workers

Sized for the task: 45 workers ÷ 8 (variable × scenario) per park ≈ 5-6 parks in flight at steady state. With ~15 parks total this keeps everyone busy.

In [ ]:
with Timer("cluster spin-up") as cluster_timer:
    cluster = coiled.Cluster(
        name="flex-ca-temp",
        region="us-central1",
        n_workers=45,
        worker_vm_types=["n2-highmem-4"],
        spot_policy="spot_with_fallback",
        idle_timeout="30 minutes",
        package_sync=True,
    )
    client = cluster.get_client()

n_workers_actual = len(client.scheduler_info()['workers'])
print(f"  Workers ready: {n_workers_actual} / 45 requested")
print(f"  Dashboard: {client.dashboard_link}")

## 4. Map-reduce across all parks — T_Max + T_Min together

Each `get_climate_data` call dispatches **2 variables × 4 scenarios = 8 tasks** to Dask per park. ThreadPoolExecutor runs the per-park calls concurrently so the scheduler sees `8 × len(ready)` tasks at once, free to schedule across all 45 workers.

`try/except` around each fetch so one park failing (network hiccup, rare edge case in the grid) doesn't abort the whole batch.

In [ ]:
scenarios = ["Historical Climate", "SSP 2-4.5", "SSP 3-7.0", "SSP 5-8.5"]
variables = ["T_Max", "T_Min"]

def fetch(name):
    t0 = perf_counter()
    try:
        data = get_climate_data(
            variables=variables, scenarios=scenarios,
            boundary=boundaries[name], time_slice=(1950, 2100),
            timescale="monthly", backend="coiled",
            coiled_cluster=cluster, catalog=cat,
        )
        # Merge T_Max + T_Min on shared keys so each row has both
        merge_keys = ["simulation", "time", "scenario", "timescale"]
        merged = data["T_Max"].merge(data["T_Min"][merge_keys + ["T_Min"]], on=merge_keys)
        merged["park"] = name
        return name, merged, perf_counter() - t0, None
    except Exception as e:
        return name, None, perf_counter() - t0, f"{type(e).__name__}: {e}"


results, errors, per_park_times = {}, {}, {}

with Timer("all-park fetch (wall-clock)") as fetch_timer:
    with ThreadPoolExecutor(max_workers=len(ready)) as ex:
        futures = [ex.submit(fetch, name) for name in ready]
        for i, f in enumerate(as_completed(futures), 1):
            name, df, dt, err = f.result()
            per_park_times[name] = dt
            if err is None:
                results[name] = df
                print(f"  [{i:2d}/{len(ready)}] {name[:45]:45s}  {dt:5.1f}s  {len(df):>7,} rows")
            else:
                errors[name] = err
                print(f"  [{i:2d}/{len(ready)}] {name[:45]:45s}  {dt:5.1f}s  ✗ {err[:70]}")

if errors:
    print(f"\n⚠️  {len(errors)} park(s) failed — details above")

## 5. Reduce — concat + compute T_Avg

In [ ]:
with Timer("concat + T_Avg") as concat_timer:
    all_temp = pd.concat(results.values(), ignore_index=True)
    all_temp["time"] = pd.to_datetime(all_temp["time"])
    all_temp["T_Avg"] = (all_temp["T_Max"] + all_temp["T_Min"]) / 2

print(f"  rows:       {len(all_temp):,}")
print(f"  parks:      {all_temp['park'].nunique()}")
print(f"  years:      {all_temp['time'].dt.year.min()}–{all_temp['time'].dt.year.max()}")
print(f"  scenarios:  {sorted(all_temp['scenario'].unique())}")
print(f"  simulations: {all_temp['simulation'].nunique()}")
print(f"  memory:     {all_temp.memory_usage(deep=True).sum() / 1e6:.0f} MB")
print(f"  T_Avg range: {all_temp['T_Avg'].min():.1f}°C to {all_temp['T_Avg'].max():.1f}°C")
all_temp.head()

## 6. Quick look — historical mean annual T_Max per park

One bar per park: multi-model-mean annual mean T_Max over 1985–2014, ranked low-to-high. Sanity check — coastal/high-elevation parks cool, desert parks hot.

In [ ]:
hist = all_temp.query("scenario == 'Historical Climate'").copy()
hist["year"] = hist["time"].dt.year
hist = hist[hist["year"].between(1985, 2014)]

# Mean annual T_Max: mean over months within each (park, simulation, year), then mean over models + years
annual = hist.groupby(["park", "simulation", "year"])["T_Max"].mean().reset_index()
park_mean = annual.groupby("park")["T_Max"].mean().sort_values()

fig, ax = plt.subplots(figsize=(10, max(4, 0.35 * len(park_mean))))
park_mean.plot.barh(ax=ax, color="#e07856")
ax.set_xlabel("Mean annual T_Max (°C), 1985–2014")
ax.set_ylabel("")
ax.set_title("California NPS units, ranked by historical mean T_Max")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()

## 7. Cost + data-volume summary

How much did the cloud actually chomp through, and how long would the same thing have taken locally? The data-volume estimate leans on what we know about LOCA2 Zarr layout (time-chunked with full spatial extent per chunk, so any time-subset fetch effectively pulls whole stores) and the timing you just watched print out above.

In [ ]:
T_TOTAL = perf_counter() - T_START

# -- Data volume (LOCA2 chunk-layout-aware estimate) --
# LOCA2 monthly 3km stores are typically chunked by time with the full western-US
# grid per chunk. A time-subset fetch therefore pulls roughly the entire store.
N_MODELS     = 15
N_VARS       = len(variables)          # 2: T_Max, T_Min
N_SCENARIOS  = len(scenarios)          # 4
N_UNIQUE_STORES = N_MODELS * N_VARS * N_SCENARIOS
AVG_STORE_GB    = 1.26   # ~1.0 GB historical + ~1.33 GB per SSP, averaged
store_reads     = N_UNIQUE_STORES * len(results)       # one full read per (store × park)
data_chomped_gb = store_reads * AVG_STORE_GB

# -- Fetch timing stats (successes only) --
success_times = {k: v for k, v in per_park_times.items() if k in results}
t_slowest = max(success_times.values()) if success_times else 0
t_fastest = min(success_times.values()) if success_times else 0
t_avg     = sum(success_times.values()) / max(len(success_times), 1)
t_serial  = sum(success_times.values())

# -- Local-equivalent estimates --
MAC_MBPS = 50                # typical home wifi sustained, MB/s
mac_hours = (data_chomped_gb * 1000) / (MAC_MBPS * 3600)
LINUX_MBPS = 200             # Linux workstation on fiber with async
linux_hours = (data_chomped_gb * 1000) / (LINUX_MBPS * 3600)

print("=" * 58)
print("  CLOUD COST / THROUGHPUT")
print("=" * 58)
print(f"  Parks processed:             {len(results)} (failed: {len(errors)}, skipped: {len(skipped)})")
print(f"  Workers actually up:         {n_workers_actual} / 45")
print(f"  Variables:                   {variables}")
print(f"  Scenarios:                   {len(scenarios)}")
print(f"  Unique Zarr stores:          {N_UNIQUE_STORES}  ({N_MODELS} models × {N_VARS} vars × {N_SCENARIOS} scenarios)")
print(f"  Store-read operations:       {store_reads:,}")
print(f"  Data chomped through:        ~{data_chomped_gb:,.0f} GB  (~{data_chomped_gb/1000:.2f} TB)")
print(f"  Rows returned to notebook:   {len(all_temp):,}")
print(f"  In-memory footprint:         {all_temp.memory_usage(deep=True).sum() / 1e6:.0f} MB")
print()
print("=" * 58)
print("  TIMING")
print("=" * 58)
print(f"  Cluster spin-up:             {cluster_timer.elapsed:6.1f}s  ({cluster_timer.elapsed/60:4.1f} min)")
print(f"  Fetch wall-clock (parallel): {fetch_timer.elapsed:6.1f}s  ({fetch_timer.elapsed/60:4.1f} min)")
if success_times:
    print(f"    ↳ avg per-park fetch:      {t_avg:6.1f}s")
    print(f"    ↳ slowest park:            {t_slowest:6.1f}s  ({max(success_times, key=success_times.get)})")
    print(f"    ↳ fastest park:            {t_fastest:6.1f}s  ({min(success_times, key=success_times.get)})")
    print(f"    ↳ sum if run sequentially: {t_serial:6.1f}s  ({t_serial/60:4.1f} min) — parallelism saved {t_serial-fetch_timer.elapsed:.0f}s")
print(f"  Concat + T_Avg:              {concat_timer.elapsed:6.1f}s")
print(f"  Total wall-clock:            {T_TOTAL:6.1f}s  ({T_TOTAL/60:4.1f} min)")
print()
print("=" * 58)
print("  LOCAL-EQUIVALENT (download bottleneck only)")
print("=" * 58)
print(f"  MacBook on home wifi ({MAC_MBPS} MB/s):       ~{mac_hours:4.1f} hours  ({mac_hours/24:.1f} days)")
print(f"  Linux + fiber + async ({LINUX_MBPS} MB/s):    ~{linux_hours:4.1f} hours")
print(f"  — both would then add single-threaded xarray processing time on top.")
print()
print(f"  Cloud speedup vs MacBook:     ~{(mac_hours*3600) / T_TOTAL:.0f}×")

## 8. Shut down

In [ ]:
cluster.close()
print("Done.")